# Caso 1: Hola Spark Streaming

Aplicación de conteo de palabras en tiempo real mediante Apache Spark Structured Streaming conectada a un socket TCP local en el puerto 9999.

In [1]:
import os
import sys
import socket
import threading
import time

jdk_path = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.21.10002-hotspot"
if os.path.exists(jdk_path):
    os.environ["JAVA_HOME"] = jdk_path
    os.environ["PATH"] = jdk_path + r"\bin;" + os.environ.get("PATH", "")
os.environ["PYSPARK_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, split

spark = SparkSession \
    .builder \
    .appName("Streaming IABD WordCount") \
    .master("local[2]") \
    .config("spark.sql.shuffle.partitions", "2") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

In [2]:
# Servidor TCP auxiliar en segundo plano para simular nc -lk 9999 directamente desde el cuaderno
def iniciar_servidor_netcat(puerto=9999):
    frases = [
        "Esta es la primera frase",
        "Enviamos segunda linea primera"
    ]
    servidor = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    servidor.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    servidor.bind(("localhost", puerto))
    servidor.listen(1)
    conn, _ = servidor.accept()
    time.sleep(2)
    for frase in frases:
        conn.sendall((frase + "\n").encode("utf-8"))
        time.sleep(3)
    time.sleep(3)
    conn.close()
    servidor.close()

hilo_nc = threading.Thread(target=iniciar_servidor_netcat, args=(9999,), daemon=True)
hilo_nc.start()

# Creamos un flujo de escucha sobre el socket en localhost:9999
# En Spark Streaming, la lectura se realiza mediante readStream
lineasDF = spark.readStream \
    .format("socket") \
    .option("host", "localhost") \
    .option("port", "9999") \
    .load()

# Leemos las líneas y las pasamos a palabras.
# Sobre ellas, realizamos la agrupación count
palabrasDF = lineasDF.select(explode(split(lineasDF.value, ' ')).alias('palabra'))
cantidadDF = palabrasDF.groupBy("palabra").count()

In [3]:
# Mostramos las palabras por consola
# En Spark Streaming, la persistencia se realiza mediante writeStream
# y en vez de hacer un save, ahora utilizamos start
wordCountQuery = cantidadDF.writeStream \
    .format("console") \
    .outputMode("complete") \
    .start()

# Mantenemos la escucha durante la recepción de los lotes de prueba
wordCountQuery.awaitTermination(timeout=10)
wordCountQuery.stop()

-------------------------------------------
Batch: 0
-------------------------------------------
+-------+-----+
|palabra|count|
+-------+-----+
+-------+-----+

-------------------------------------------
Batch: 1
-------------------------------------------
+-------+-----+
|palabra|count|
+-------+-----+
|     es|    1|
|  frase|    1|
|   Esta|    1|
|     la|    1|
|primera|    1|
+-------+-----+

-------------------------------------------
Batch: 2
-------------------------------------------
+--------+-----+
| palabra|count|
+--------+-----+
|   linea|    1|
|      es|    1|
|Enviamos|    1|
| segunda|    1|
|      la|    1|
| primera|    2|
|   frase|    1|
|    Esta|    1|
+--------+-----+
